In [3]:
from pymongo import MongoClient

client = MongoClient("mongodb://bigdata_mongodb:27017/", serverSelectionTimeoutMS=5000)

print("Conectando...")
print(client.server_info())  

db = client["Inmobiliaria"]
coleccion = db["test"]

coleccion.insert_one({"ok": "funciona"})
print("Insertado correctamente")

Conectando...
{'version': '8.2.6', 'versionArray': [8, 2, 6, 0], 'gitVersion': '5d25c835745d06f712320b6cdae9d50b7b43663e', 'modules': [], 'allocator': 'tcmalloc-google', 'javascriptEngine': 'mozjs', 'sysinfo': 'deprecated', 'openssl': {'running': 'OpenSSL 3.0.13 30 Jan 2024', 'compiled': 'OpenSSL 3.0.13 30 Jan 2024'}, 'buildEnvironment': {'distmod': 'ubuntu2404', 'distarch': 'x86_64', 'cc': 'external/mongo_toolchain_v5/v5/bin/gcc: gcc (GCC) 14.2.0\nCopyright (C) 2024 Free Software Foundation, Inc.\nThis is free software; see the source for copying conditions.  There is NO\nwarranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.\n\n', 'ccflags': '-U_FORTIFY_SOURCE -std=c++20 -nostdinc++ -Wall -Wsign-compare -Winvalid-pch -Wno-missing-braces -Wno-psabi -Bexternal/mongo_toolchain_v5/v5/bin -Bexternal/mongo_toolchain_v5/stow/gcc-v5/libexec/gcc/x86_64-mongodb-linux/14.2.0 -Bexternal/mongo_toolchain_v5/stow/gcc-v5/lib/gcc/x86_64-mongodb-linux/14.2.0 -Bexternal/mongo_toolc

In [8]:
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os

# =========================
# LIMPIEZA (evita errores de Chrome)
# =========================
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")

# =========================
# CONEXIÓN MONGO
# =========================
client = MongoClient("mongodb://bigdata_mongodb:27017/")
db = client["Inmobiliaria"]
coleccion = db["propiedades"]

# =========================
# CONFIGURACIÓN SELENIUM (DOCKER)
# =========================
options = Options()
options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--headless=new")  # 🔥 modo seguro
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

# =========================
# NAVEGACIÓN
# =========================
url = "https://www.portalinmobiliario.com/arriendo/departamento/coquimbo-coquimbo"
driver.get(url)

WebDriverWait(driver, 20).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
)

# =========================
# SCROLL (cargar más resultados)
# =========================
for _ in range(3):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")

print("Total encontrados:", len(bloques))

datos = []

# =========================
# SCRAPING
# =========================
for b in bloques:
    try:
        titulo = b.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
        precio = b.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

        if not titulo or not precio:
            continue

        # 🔹 LIMPIEZA PRECIO
        precio_limpio = precio.replace("$", "").replace(".", "").strip()
        precio_num = float(precio_limpio) if precio_limpio.isdigit() else 0.0

        # 🔹 EXTRA OPCIONAL
        try:
            ubicacion = b.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")
        except:
            ubicacion = "No disponible"

        try:
            caracteristicas = b.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")
        except:
            caracteristicas = "No disponible"

        data = {
            "titulo": titulo.strip(),
            "precio": precio_num,
            "precio_texto": precio.strip(),
            "ubicacion": ubicacion.strip(),
            "caracteristicas": caracteristicas.strip(),
            "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
            "grupo": "G2_Inmobiliaria"
        }

        datos.append(data)

        print("----")
        print("Título:", titulo.strip())
        print("Precio:", precio.strip())

    except:
        continue

# =========================
# GUARDAR EN MONGO
# =========================
if datos:
    coleccion.insert_many(datos)
    print(f"\n {len(datos)} propiedades guardadas en MongoDB")
else:
    print("⚠️ No se encontraron datos válidos")

# =========================
# CIERRE
# =========================
driver.quit()

Total encontrados: 48
----
Título: Arriendo Centro De Coquimbo: Depto Amoblado Con Vista Al Mar
Precio: $450.000
----
Título: Se Arrienda Departamento Marzo A Diciembre Costa Elqui
Precio: $650.000
----
Título: Edificio Manuel Jesús Rivera (1 Est. 1 Bod.) Vista Sur Pis
Precio: $390.000
----
Título: Arriendo Hermoso Penthouse A Pasos De Av Mar (148150)
Precio: $1.500.000
----
Título: Arriendo Departamento Amoblado, La  Herradura
Precio: $650.000
----
Título: Arriendo Departamento Amoblado Detras Casino Enjoy Coquimbo
Precio: $630.000
----
Título: Moderno Depto Amoblado Primer Piso En Av Los Lagos
Precio: $850.000
----
Título: Departamento Amoblado A Pasos De Playa
Precio: $600.000
----
Título: Depto Amoblado Primer Piso En Condominio Alto Hacienda
Precio: $580.000
----
Título: Se Arrienda Departamento Amoblado Jardín Del Mar
Precio: $700.000
----
Título: Departamento En Arriendo, 3 Dorm Bosque San Carlos,coquimbo
Precio: $470.000
----
Título: Nuevo, Amoblado, Año Corrido Sector Casino E